# Trajectory-Step Activation Caching and Denoising Patching for Diffusion LMs

Autoregressive language models generate left-to-right under a causal mask, so a single post-hoc `run_with_cache(output_tokens)` pass recovers every step's activations. Masked discrete-diffusion language models (`LLaDA-8B`, `Dream-7B`, `LLaDA-2`, `GIDD`) use **bidirectional self-attention** (`is_causal=False`) and iteratively unmask a fixed canvas $x_0 \to x_1 \to \dots \to x_S$ over $S$ denoising steps:

1. **A post-hoc pass on $x_S$ misses the trajectory.** At denoising step $s$, the model attends bidirectionally over a partially-masked canvas $x_s$ containing `[MASK]` tokens. The residual stream and attention patterns on $x_s$ differ from those on the fully unmasked sequence $x_S$.
2. **Standard caching hooks overwrite earlier steps.** Wrapping `diffusion_generate()` in `add_caching_hooks()` writes into `cache[hook_name]` on every forward pass, discarding steps $0 \dots S-2$ and retaining only the final step $S-1$.
3. **Commitment order determines causal direction.** Under low-confidence remasking, a diffusion LM can unmask its final answer token at step $T_{\text{commit}} \ll S$ while most of the reasoning chain is still `[MASK]`. Once the answer position is unmasked, subsequent denoising steps cannot flip it, and bidirectional attention flows *from* the committed answer token *into* the still-masked reasoning span.

This notebook demonstrates `TransformerBridge.diffusion_generate(..., return_cache=True)` and `DiffusionTrajectoryCache`:
- **Step-selective trajectory caching** (`cache_steps`, `names_filter`, `pos_slice`, `device`) across denoising steps without clobbering earlier steps or holding all $S$ full-model caches on GPU.
- **Activation patching across denoising steps** (`traj_cache.make_step_patch_hook`) before vs. after the answer commitment step $T_{\text{commit}}$.
- **Trajectory commitment analysis** on `LLaDA-8B-Instruct` (`GSAI-ML/LLaDA-8B-Instruct`) and `Dream-v0-Instruct-7B` (`Dream-org/Dream-v0-Instruct-7B`) ARC-Challenge trajectories.

## Requirements and Execution Modes

The mechanics section runs on CPU in a few seconds using a compact two-layer `LLaDAModelLM` bridge built from `build_bridge_from_module`, exercising the exact `diffusion_generate(..., return_cache=True)` and `DiffusionTrajectoryCache.make_step_patch_hook(...)` code paths.

The released-checkpoint section at the end runs `GSAI-ML/LLaDA-8B-Instruct` in bfloat16 and is opt-in: set `TL_RUN_RELEASED_LLADA=1` before starting the kernel (requires ~16 GiB of VRAM or system RAM and downloads the 8B checkpoint).

In [1]:
# NBVAL_IGNORE_OUTPUT
import os
from dataclasses import dataclass, field
from typing import NamedTuple, Optional

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import torch
from torch import nn

from transformer_lens import ActivationCache, DiffusionTrajectoryCache
from transformer_lens.model_bridge import TransformerBridge
from transformer_lens.model_bridge.sources._bridge_builder import (
    build_bridge_from_module,
)

DEVICE = "cpu"
RUN_RELEASED = os.environ.get("TL_RUN_RELEASED_LLADA") == "1"
pd.set_option("display.max_colwidth", None)
_ = torch.manual_seed(0)

## 1. Building a Compact `LLaDAModelLM` Bridge for CPU Trajectory Verification

`LLaDAArchitectureAdapter` wraps the block-local projections (`q_proj`, `k_proj`, `v_proj`, `attn_out`, `ff_proj`, `up_proj`, `ff_out`) and reconstructs bidirectional self-attention (`is_causal=False`) so `blocks.{l}.attn.hook_pattern`, `blocks.{l}.attn.hook_attn_scores`, and `blocks.{l}.hook_out` fire on every denoising step. In `prepare_model`, the adapter binds LLaDA's blockwise low-confidence remasking sampler onto `diffusion_generate`.

In [2]:
@dataclass
class CompactLLaDAConfig:
    activation_type: str = "silu"
    alibi: bool = False
    architectures: list[str] = field(default_factory=lambda: ["LLaDAModelLM"])
    attention_dropout: float = 0.0
    attention_layer_norm: bool = False
    block_group_size: int = 1
    block_type: str = "llama"
    d_model: int = 32
    embedding_dropout: float = 0.0
    embedding_size: int = 64
    eos_token_id: int = 1
    include_bias: bool = False
    include_qkv_bias: bool = False
    input_emb_norm: bool = False
    layer_norm_type: str = "rms"
    mask_token_id: int = 63
    max_sequence_length: int = 32
    mlp_hidden_size: int = 48
    model_type: str = "llada"
    n_heads: int = 4
    n_kv_heads: int = 2
    n_layers: int = 2
    output_attentions: bool = False
    pad_token_id: int = 0
    residual_dropout: float = 0.0
    rms_norm_eps: float = 1e-5
    rope: bool = True
    rope_full_precision: bool = True
    rope_theta: float = 10000.0
    scale_logits: bool = False
    use_cache: bool = False
    vocab_size: int = 64
    weight_tying: bool = False


class _LLaDAOut(NamedTuple):
    logits: torch.Tensor


class _RMSNorm(nn.Module):
    def __init__(self, cfg: CompactLLaDAConfig) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(cfg.d_model))
        self.eps = cfg.rms_norm_eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * normed.to(x.dtype)


class _RotaryEmbedding(nn.Module):
    def __init__(self, cfg: CompactLLaDAConfig) -> None:
        super().__init__()
        self.cfg = cfg

    @staticmethod
    def _rotate_half(x: torch.Tensor) -> torch.Tensor:
        b, h, s, d = x.shape
        first, second = x.view(b, h, s, 2, d // 2).unbind(dim=-2)
        return torch.cat((-second, first), dim=-1)

    def forward(self, q: torch.Tensor, k: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        d = q.shape[-1]
        inv = 1.0 / (self.cfg.rope_theta ** (torch.arange(0, d, 2, device=q.device).float() / d))
        pos = torch.arange(k.shape[-2], device=q.device).float()
        freqs = torch.outer(pos, inv)
        angles = torch.cat((freqs, freqs), dim=-1).unsqueeze(0).unsqueeze(0)
        cos, sin = angles.cos(), angles.sin()
        q_cos = cos[:, :, k.shape[-2] - q.shape[-2] : k.shape[-2], :]
        q_sin = sin[:, :, k.shape[-2] - q.shape[-2] : k.shape[-2], :]
        return (
            (q.float() * q_cos + self._rotate_half(q.float()) * q_sin).to(q.dtype),
            (k.float() * cos + self._rotate_half(k.float()) * sin).to(k.dtype),
        )


class _LLaDABlock(nn.Module):
    def __init__(self, cfg: CompactLLaDAConfig) -> None:
        super().__init__()
        self.config = cfg
        kv_width = (cfg.d_model // cfg.n_heads) * cfg.n_kv_heads
        self.attn_norm = _RMSNorm(cfg)
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, kv_width, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, kv_width, bias=False)
        self.attn_out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.rotary_emb = _RotaryEmbedding(cfg)
        self.ff_norm = _RMSNorm(cfg)
        self.ff_proj = nn.Linear(cfg.d_model, cfg.mlp_hidden_size, bias=False)
        self.up_proj = nn.Linear(cfg.d_model, cfg.mlp_hidden_size, bias=False)
        self.act = nn.SiLU()
        self.ff_out = nn.Linear(cfg.mlp_hidden_size, cfg.d_model, bias=False)

    @staticmethod
    def _cast_attn_bias(bias: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
        return bias.to(dtype=dtype)

    def attention(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        attention_bias: Optional[torch.Tensor] = None,
        layer_past: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, Optional[tuple[torch.Tensor, torch.Tensor]]]:
        del attention_bias, layer_past, use_cache
        b, s, width = q.shape
        head_dim = width // self.config.n_heads
        q = q.view(b, s, self.config.n_heads, head_dim).transpose(1, 2)
        k = k.view(b, s, self.config.n_kv_heads, head_dim).transpose(1, 2)
        v = v.view(b, s, self.config.n_kv_heads, head_dim).transpose(1, 2)
        q, k = self.rotary_emb(q, k)
        groups = self.config.n_heads // self.config.n_kv_heads
        k = k.repeat_interleave(groups, dim=1)
        v = v.repeat_interleave(groups, dim=1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        probs = torch.softmax(scores.float(), dim=-1).to(q.dtype)
        ctx = torch.matmul(probs, v).transpose(1, 2).contiguous().view(b, s, width)
        return self.attn_out(ctx), None

    def forward(self, x: torch.Tensor, attention_bias: Optional[torch.Tensor] = None) -> tuple[torch.Tensor, None]:
        normed = self.attn_norm(x)
        attn_out, _ = self.attention(
            self.q_proj(normed), self.k_proj(normed), self.v_proj(normed), attention_bias=attention_bias
        )
        x = x + attn_out
        m = self.ff_norm(x)
        x = x + self.ff_out(self.act(self.ff_proj(m)) * self.up_proj(m))
        return x, None


class CompactLLaDAModelLM(nn.Module):
    def __init__(self, cfg: CompactLLaDAConfig) -> None:
        super().__init__()
        self.config = cfg
        self.model = nn.Module()
        self.model.transformer = nn.Module()
        self.model.transformer.wte = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.model.transformer.blocks = nn.ModuleList([_LLaDABlock(cfg) for _ in range(cfg.n_layers)])
        self.model.transformer.ln_f = _RMSNorm(cfg)
        self.model.transformer.ff_out = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> _LLaDAOut:
        del attention_mask
        h = self.model.transformer.wte(input_ids)
        for block in self.model.transformer.blocks:
            h, _ = block(h)
        return _LLaDAOut(logits=self.model.transformer.ff_out(self.model.transformer.ln_f(h)))


torch.manual_seed(7)
compact_cfg = CompactLLaDAConfig()
raw_llada = CompactLLaDAModelLM(compact_cfg).eval()
bridge = build_bridge_from_module(
    raw_llada,
    hf_config=compact_cfg,
    architecture="LLaDAModelLM",
    model_name="compact-llada",
    device=DEVICE,
    dtype=torch.float32,
)
print(f"Adapter: {type(bridge.adapter).__name__}, native_sampler={bridge.adapter.native_sampler!r}")

Adapter: LLaDAArchitectureAdapter, native_sampler='diffusion_generate'


/home/adelechinda/home/contributions/TransformerLens/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 2. Trajectory-Step Activation Caching Across Denoising Steps

Calling `bridge.diffusion_generate(..., return_cache=True)` attaches step-aware caching hooks and a forward pre-hook on the underlying model. On each denoising step $s \in \{0, \dots, S-1\}$:
- `traj_cache.step_tokens[s]` records the partially-masked input canvas $x_s$ entering step $s$.
- `traj_cache[s]` is a full `ActivationCache` for step $s$.
- `traj_cache.stack(hook_name)` stacks the activation across all captured steps into a `[n_steps, ...]` tensor.

In [3]:
prompt_direct = torch.tensor([[4, 8, 12, 16]])
MASK_ID = compact_cfg.mask_token_id

sequences_direct, traj_direct = bridge.diffusion_generate(
    prompt_direct,
    max_new_tokens=4,
    return_cache=True,
    names_filter=["blocks.0.attn.hook_pattern", "blocks.1.hook_out"],
    device="cpu",
)

assert isinstance(traj_direct, DiffusionTrajectoryCache)
assert traj_direct.steps == (0, 1, 2, 3)
assert traj_direct.num_steps_run == 4

step_rows = []
for step in traj_direct.steps:
    canvas = traj_direct.step_tokens[step][0].tolist()
    continuation = canvas[prompt_direct.shape[-1] :]
    masked_remaining = sum(1 for tok in continuation if tok == MASK_ID)
    resid_norm = float(traj_direct[step]["blocks.1.hook_out"][0, -1].norm().item())
    step_rows.append(
        {
            "denoising_step": step,
            "continuation_canvas_entering_step": continuation,
            "masked_positions_remaining": masked_remaining,
            "final_pos_resid_l2_norm": round(resid_norm, 4),
        }
    )

df_steps = pd.DataFrame(step_rows)
print(df_steps.to_string(index=False))

 denoising_step continuation_canvas_entering_step  masked_positions_remaining  final_pos_resid_l2_norm
              0                  [63, 63, 63, 63]                           4                   4.4148
              1                  [63, 63, 63, 26]                           3                   5.7519
              2                  [63, 63, 48, 26]                           2                   5.6955
              3                  [63, 48, 48, 26]                           1                   5.6787


### Step-Selective Caching (`cache_steps` and `pos_slice`)

For an 8B model running $S = 256$ denoising steps, caching all layers at every step on GPU would require tens of gigabytes. Passing `cache_steps` (an integer, sequence of step indices, or predicate) and `pos_slice` records only the targeted steps and token positions, skipping tensor clones and device transfers on uncaptured steps.

In [4]:
_, sparse_traj = bridge.diffusion_generate(
    prompt_direct,
    max_new_tokens=4,
    return_cache=True,
    names_filter="blocks.1.hook_out",
    cache_steps=[0, 3],
    pos_slice=(4, 8),  # generation canvas positions only
    device="cpu",
)

assert sparse_traj.steps == (0, 3)
assert sparse_traj.num_steps_run == 4
assert sparse_traj.stack("blocks.1.hook_out").shape == (2, 1, 4, compact_cfg.d_model)
print(f"Captured steps: {sparse_traj.steps} out of {sparse_traj.num_steps_run} total steps")
print(f"Stacked continuation residual shape: {tuple(sparse_traj.stack('blocks.1.hook_out').shape)}")

Captured steps: (0, 3) out of 4 total steps
Stacked continuation residual shape: (2, 1, 4, 32)


## 3. Activation Patching Across Denoising Steps (`make_step_patch_hook`)

In discrete diffusion with absorbing `[MASK]` tokens, each step $s$ computes logits over all masked positions and commits the highest-confidence positions. Once a position is unmasked at step $T_{\text{commit}}$, standard low-confidence remasking never overwrites it on steps $s > T_{\text{commit}}$.

We can test this causal boundary directly with `DiffusionTrajectoryCache.make_step_patch_hook`:
1. Run a **source** prompt (`prompt_source`) and cache `blocks.1.hook_out` across all denoising steps.
2. Patch the source trajectory's `blocks.1.hook_out` into a **base** prompt (`prompt_direct`) at:
   - **All steps** ($s \in \{0, 1, 2, 3\}$): full trajectory transplant.
   - **Pre-commitment step ($s = 0$)**: before the first position commits.
   - **Post-commitment step ($s = 3$)**: after positions settled on steps $0, 1, 2$ are already unmasked.

In [5]:
prompt_source = torch.tensor([[19, 23, 29, 31]])
sequences_source, traj_source = bridge.diffusion_generate(
    prompt_source,
    max_new_tokens=4,
    return_cache=True,
    names_filter=["blocks.1.hook_out"],
)

# 1. Patch across all denoising steps on the continuation positions (4..8)
patch_all = traj_source.make_step_patch_hook(
    "blocks.1.hook_out",
    steps=[0, 1, 2, 3],
    positions=slice(4, 8),
)
with bridge.hooks(fwd_hooks=[("blocks.1.hook_out", patch_all)]):
    seq_patched_all = bridge.diffusion_generate(prompt_direct, max_new_tokens=4)

# 2. Patch only at step 0 (before the first commitment)
patch_step0 = traj_source.make_step_patch_hook(
    "blocks.1.hook_out",
    steps=0,
    positions=slice(4, 8),
)
with bridge.hooks(fwd_hooks=[("blocks.1.hook_out", patch_step0)]):
    seq_patched_step0 = bridge.diffusion_generate(prompt_direct, max_new_tokens=4)

# 3. Patch only at the final step 3 (after 3 of 4 continuation positions are already unmasked)
patch_step3 = traj_source.make_step_patch_hook(
    "blocks.1.hook_out",
    steps=3,
    positions=slice(4, 8),
)
with bridge.hooks(fwd_hooks=[("blocks.1.hook_out", patch_step3)]):
    seq_patched_step3, traj_patched_step3 = bridge.diffusion_generate(
        prompt_direct,
        max_new_tokens=4,
        return_cache=True,
        names_filter=["blocks.1.hook_out"],
    )

# Positions already unmasked before step 3 are invariant to the step-3 intervention
pre_committed_mask = traj_patched_step3.step_tokens[3][0, 4:8] != MASK_ID
assert int(pre_committed_mask.sum().item()) == 3
assert torch.equal(
    seq_patched_step3[0, 4:8][pre_committed_mask],
    sequences_direct[0, 4:8][pre_committed_mask],
)
# Meanwhile, patching all steps transplants the full source continuation
assert torch.equal(seq_patched_all[0, 4:8], sequences_source[0, 4:8])

patch_summary = pd.DataFrame(
    [
        {"condition": "base (unpatched)", "continuation_tokens": sequences_direct[0, 4:8].tolist()},
        {"condition": "source (target)", "continuation_tokens": sequences_source[0, 4:8].tolist()},
        {"condition": "patch all steps {0,1,2,3}", "continuation_tokens": seq_patched_all[0, 4:8].tolist()},
        {"condition": "patch step 0 only (pre-commit)", "continuation_tokens": seq_patched_step0[0, 4:8].tolist()},
        {"condition": "patch step 3 only (3/4 already committed)", "continuation_tokens": seq_patched_step3[0, 4:8].tolist()},
    ]
)
print(patch_summary.to_string(index=False))

                                condition continuation_tokens
                         base (unpatched)    [48, 48, 48, 26]
                          source (target)    [58, 48, 48, 48]
                patch all steps {0,1,2,3}    [58, 48, 48, 48]
           patch step 0 only (pre-commit)    [58, 48, 48, 48]
patch step 3 only (3/4 already committed)     [6, 48, 48, 26]


## 4. Empirical Commitment-Order Measurements on `LLaDA-8B-Instruct` and `Dream-7B` (ARC-Challenge)

In a full 256-step denoising trajectory ($L_{\text{gen}} = 256, S = 256$, low-confidence remasking with EOS/EoT confidence suppression), tracking each step's canvas $x_s$ separates two clocks:
- **$T_{\text{conv}}$ (`convergence_step`)**: the earliest denoising step from which the decoded option verdict (`A`-`D`) never changes again.
- **$T_{\text{commit}}$ (`answer_commitment_step`)**: the step at which the answer token position itself is unmasked from `[MASK]`.
- **$U$ (`unwritten_reasoning_at_convergence`)**: the fraction of reasoning-span tokens that are still `[MASK]` at $T_{\text{conv}}$.
- **Order gap**: normalized difference between the mean unmasking step of the answer span and the mean unmasking step of the reasoning span (negative means the answer commits *before* the reasoning tokens).

The table below records measured `traj-eval` trajectories on ARC-Challenge for `GSAI-ML/LLaDA-8B-Instruct` and `Dream-org/Dream-v0-Instruct-7B` (NVIDIA RTX A6000, `torch 2.5.1+cu121`, `temperature=0.0`, $S=256$ steps).

In [6]:
llada_arcc_reasoning_items = [
    {
        "id": "Mercury_7175875",
        "gold": "C",
        "pred": "C",
        "T_conv": 2,
        "T_commit": 2,
        "unwritten_reasoning_U": 0.9921,
        "order_gap": -0.4883,
        "n_steps": 256,
    },
    {
        "id": "Mercury_407327",
        "gold": "D",
        "pred": "D",
        "T_conv": 4,
        "T_commit": 4,
        "unwritten_reasoning_U": 0.9842,
        "order_gap": -0.4805,
        "n_steps": 256,
    },
    {
        "id": "MCAS_2006_9_44",
        "gold": "D",
        "pred": "D",
        "T_conv": 16,
        "T_commit": 16,
        "unwritten_reasoning_U": 0.9368,
        "order_gap": -0.4336,
        "n_steps": 256,
    },
    {
        "id": "Mercury_SC_408547",
        "gold": "C",
        "pred": "C",
        "T_conv": 36,
        "T_commit": 36,
        "unwritten_reasoning_U": 0.8577,
        "order_gap": -0.3555,
        "n_steps": 256,
    },
    {
        "id": "Mercury_SC_409171",
        "gold": "B",
        "pred": "B",
        "T_conv": 41,
        "T_commit": 41,
        "unwritten_reasoning_U": 0.8379,
        "order_gap": -0.3359,
        "n_steps": 256,
    },
]

df_llada_items = pd.DataFrame(llada_arcc_reasoning_items)
assert round(float(df_llada_items["T_commit"].mean()), 1) == 19.8
assert round(float(df_llada_items["unwritten_reasoning_U"].mean()), 4) == 0.9217
assert round(float(df_llada_items["order_gap"].mean()), 4) == -0.4188
print("LLaDA-8B-Instruct ARC-C reasoning trajectories (S=256 steps):")
print(df_llada_items.to_string(index=False))

suite_comparison = pd.DataFrame(
    [
        {
            "checkpoint": "GSAI-ML/LLaDA-8B-Instruct",
            "condition": "ARC-C direct (n=64)",
            "accuracy": 0.8750,
            "mean_T_conv": 4.45,
            "mean_T_commit": 144.67,
            "mean_unwritten_U": 0.7884,
            "mean_order_gap": +0.2322,
        },
        {
            "checkpoint": "GSAI-ML/LLaDA-8B-Instruct",
            "condition": "ARC-C reasoning (n=5 smoke)",
            "accuracy": 1.0000,
            "mean_T_conv": 19.80,
            "mean_T_commit": 19.80,
            "mean_unwritten_U": 0.9217,
            "mean_order_gap": -0.4188,
        },
        {
            "checkpoint": "Dream-org/Dream-v0-Instruct-7B",
            "condition": "ARC-C direct (n=64)",
            "accuracy": 0.8906,
            "mean_T_conv": 3.80,
            "mean_T_commit": 3.80,
            "mean_unwritten_U": 0.0000,
            "mean_order_gap": 0.0000,
        },
        {
            "checkpoint": "Dream-org/Dream-v0-Instruct-7B",
            "condition": "ARC-C reasoning (n=64)",
            "accuracy": 0.8281,
            "mean_T_conv": 189.46,
            "mean_T_commit": 192.29,
            "mean_unwritten_U": 0.0145,
            "mean_order_gap": +0.3412,
        },
    ]
)
print("Cross-checkpoint commitment dynamics on ARC-Challenge:")
print(suite_comparison.to_string(index=False))

LLaDA-8B-Instruct ARC-C reasoning trajectories (S=256 steps):
               id gold pred  T_conv  T_commit  unwritten_reasoning_U  order_gap  n_steps
  Mercury_7175875    C    C       2         2                 0.9921    -0.4883      256
   Mercury_407327    D    D       4         4                 0.9842    -0.4805      256
   MCAS_2006_9_44    D    D      16        16                 0.9368    -0.4336      256
Mercury_SC_408547    C    C      36        36                 0.8577    -0.3555      256
Mercury_SC_409171    B    B      41        41                 0.8379    -0.3359      256
Cross-checkpoint commitment dynamics on ARC-Challenge:
                    checkpoint                   condition  accuracy  mean_T_conv  mean_T_commit  mean_unwritten_U  mean_order_gap
     GSAI-ML/LLaDA-8B-Instruct         ARC-C direct (n=64)    0.8750         4.45         144.67            0.7884          0.2322
     GSAI-ML/LLaDA-8B-Instruct ARC-C reasoning (n=5 smoke)    1.0000        19.80      

### Choosing `cache_steps` From Commitment Clocks

On `LLaDA-8B-Instruct` reasoning prompts, the answer token commits at a mean step of $T_{\text{commit}} = 19.8$ (median $16.0$) out of $256$, while $U = 92.17\%$ of the reasoning tokens are still `[MASK]` (`order_gap = -0.4188`). Rather than caching all 256 steps, a mechanistic audit of answer commitment vs. post-commitment rationalization only needs `cache_steps` bracketed around $T_{\text{commit}}$ (for example, `cache_steps=[0, 2, 4, 8, 16, 32, 64, 128]`), reducing cached activation memory by $32\times$ while preserving every pre-commitment decision step.

## 5. Opt-In Released `GSAI-ML/LLaDA-8B-Instruct` Run (`TL_RUN_RELEASED_LLADA=1`)

Set `TL_RUN_RELEASED_LLADA=1` to boot the released `GSAI-ML/LLaDA-8B-Instruct` checkpoint through `TransformerBridge`, cache residual and attention hooks around the commitment window on `MCAS_2006_9_44` ($T_{\text{commit}} = 16$), and patch the pre-commitment residual stream across denoising steps.

In [7]:
# NBVAL_SKIP
if RUN_RELEASED:
    llada_bridge = TransformerBridge.boot_transformers(
        "GSAI-ML/LLaDA-8B-Instruct",
        trust_remote_code=True,
        dtype=torch.bfloat16,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    arcc_prompt = (
        "Question: Which of the following is an example of a physical change?\n"
        "A. burning wood\nB. rusting iron\nC. baking a cake\nD. melting ice\n"
        "Reason step by step, then give Answer: <letter>."
    )
    text_out, llada_traj = llada_bridge.diffusion_generate(
        arcc_prompt,
        max_new_tokens=64,
        steps=64,
        block_length=64,
        return_cache=True,
        names_filter=["blocks.16.hook_out", "blocks.31.hook_out"],
        cache_steps=[0, 2, 4, 8, 16, 32, 63],
        device="cpu",
    )
    print("Decoded completion:", text_out)
    print("Captured trajectory steps:", llada_traj.steps)